# RegimeShift: Market-Regime Asset Allocation System
## IIT Bombay Summer Quant 2026 — Final Submission

This notebook reproduces the complete RegimeShift pipeline:
data → features → HMM → portfolio → backtest → metrics → research validation.

All outputs are computed from the **real market dataset** loaded at
the top of this notebook.

**Dataset SHA-256 (canonical):** `290c783af9fb803334388090c3b72df7a31f5fd580de62bb7a905250777226e7`

**Official cost model:** base scenario — 10 bps one-way per asset

**Execution model:** NEXT_CLOSE (no same-bar fills)

### 1. Research Question

Does a 3-state Gaussian HMM improve risk-adjusted returns (Sharpe ratio) for a three-asset Indian multi-asset portfolio — NIFTY 50 equity, GOLDBEES.NS gold, LIQUIDBEES.NS bond — relative to static benchmarks, under causal NEXT_CLOSE execution and realistic transaction cost assumptions?

### 2. Asset Universe

| Role | Instrument | Ticker | Notes |
|---|---|---|---|
| Equity | NIFTY 50 Index | `^NSEI` | **Index-level simulation — not directly executable** |
| Gold | Nippon India ETF Gold Bees | `GOLDBEES.NS` | INR-denominated gold ETF |
| Defensive | Nippon India Liquid Bees | `LIQUIDBEES.NS` | Liquid-bond / cash-equivalent proxy; short duration |
| Volatility feature | India VIX | `^INDIAVIX` | Optional feature only; omitted from official submitted CSV |

**Index-level tradability limitation:** `^NSEI` is an index series, not a directly tradeable instrument. Any real implementation requires a separately verified equity ETF (e.g. Nippon India NIFTY 50 BeES). This is an **index-level allocation simulation**, not an executable portfolio.

**Bond proxy:** LIQUIDBEES.NS — Nippon India Liquid Bees, INR-denominated liquid-bond / cash-equivalent proxy (NOT a sovereign bond or 10-year G-Sec; very short duration, no long-term rate hedge)

### 3. Dataset and Canonical Hash

| Property | Value |
|---|---|
| File | `data/submission_market_data.csv` |
| SHA-256 (canonical) | `290c783af9fb803334388090c3b72df7a31f5fd580de62bb7a905250777226e7` |
| First date | `2010-01-04` |
| Last date | `2026-07-27` |
| Row count | `4090` |
| Columns | `equity, gold, bond` |
| VIX included | `False` |
| Total forward-filled cells | `0` |
| Dates dropped (residual NaNs) | `0` |

Fill provenance is recorded explicitly by the data pipeline before any forward-fill occurs. Naturally constant prices (e.g. LIQUIDBEES.NS liquid-bond NAV) are not considered fill artefacts.

Third-party market data is included only for academic reproducibility; it remains subject to the terms of its original providers and exchanges.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path(".").resolve() / "src"))

from regime_shift.config import RegimeShiftConfig
from regime_shift.execution import ExecutionCostModel
from regime_shift.data import load_market_data_csv

config = RegimeShiftConfig()
config.execution_cost_model = ExecutionCostModel.scenario("base", config.core_assets)

DATA_PATH = "data/submission_market_data.csv"
prices = load_market_data_csv(path=DATA_PATH, config=config)
diag = prices.attrs.get("data_diagnostics", {})

import hashlib
with open(DATA_PATH, "rb") as _fh:
    _raw = _fh.read().replace(b"\r\n", b"\n")
canonical_sha256 = hashlib.sha256(_raw).hexdigest()

print(f"Price data: {len(prices)} rows")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Columns: {list(prices.columns)}")
print(f"Canonical SHA-256: {canonical_sha256}")
print(f"Forward-filled cells: {diag.get('ffill_cells_total', 0)}")
print(f"Dates dropped: {diag.get('dates_dropped', 0)}")
display(prices.head())
display(prices.describe())


### 4. Causal NEXT_CLOSE Execution Timing

At close **t**:

1. Holdings established after close **t-1** earn the **t-1 → t** return.
2. Close **t** price becomes known.
3. Features, scaler, and HMM may use information strictly through **t**.
4. Regime and portfolio target are generated at close **t**.
5. Execution costs are paid at close **t** (applied to returned wealth).
6. New holdings first earn the **t → t+1** return.

This is the NEXT_CLOSE convention. No same-bar fills. No future data.

```
Prices
  ↓
Causal trailing features (rolling windows, no lookahead)
  ↓
Train-only StandardScaler (fit on rolling window ≤ t)
  ↓
Rolling 3-state Gaussian HMM (fit on rolling window ≤ t)
  ↓
Regime-conditioned CVXPY allocation
  ↓
NEXT_CLOSE execution (costs at close t, return starts t→t+1)
  ↓
Benchmark-aligned evaluation with identical cost model
```

### 5. Leakage-Safe Features

Seven base features computed from strictly trailing windows:
log returns, realized volatility (21d, 63d), price-to-MA ratios, MACD signal, rolling drawdown, and cross-asset correlations.
VIX features are optional and omitted when VIX is not in the dataset.

The StandardScaler is fitted only on the rolling training window (252 days through close **t**). No future data reaches the scaler.

In [ ]:
from regime_shift.features import (
    compute_raw_features, drop_feature_warmup,
    fit_feature_scaler, transform_features,
)

feature_cfg = config.feature_config
raw_features = compute_raw_features(prices, config=feature_cfg)
raw_features = drop_feature_warmup(raw_features, config=feature_cfg)
print(f"Features after warmup: {len(raw_features)} rows, {len(raw_features.columns)} columns")
print(f"Feature columns: {list(raw_features.columns)}")
display(raw_features.head())


### 6. Train-Only StandardScaler and 3-Restart Gaussian HMM

Training window: **252 days** rolling.
HMM: 3 states, diagonal covariance, 3 deterministic restarts per fit.

Restart selection: highest converged training log-likelihood. **No selection based on out-of-sample performance.**

In [ ]:
from regime_shift.regime_model import fit_hmm

train_end = raw_features.index[min(config.train_window, len(raw_features) - 1)]
train_features = raw_features.loc[:train_end].tail(config.train_window)
scaler = fit_feature_scaler(train_features)
scaled_train = transform_features(scaler, train_features)
hmm, hidden_states, trans_mat = fit_hmm(scaled_train, train_features, config=config.hmm_config)
print(f"HMM converged: {hmm.monitor_.converged}")
print(f"Iterations: {hmm.monitor_.n_iter}")
print(f"Log-likelihood: {hmm.score(scaled_train.values):.2f}")
print("Transition matrix:")
display(trans_mat)


### 7. Bull / Bear / Crisis State Mapping

Numeric HMM states are mapped to interpretable labels using training-period volatility, momentum, and VIX statistics only. **No out-of-sample performance is used for state labelling.**

In [ ]:
from regime_shift.regime_model import predict_current_state
solution = predict_current_state(
    hmm, scaler, train_features, train_features, config=config.hmm_config,
)
print(f"Current regime: {solution.regime}")
print(f"Regime probabilities: {dict(solution.probabilities)}")
print(f"Convergence: {solution.convergence}")
print(f"Iterations: {solution.n_iter}")


### 8. Regime-Conditioned CVXPY Portfolio Construction

Each regime has a distinct convex objective and regime-specific weight constraints:

| Regime | Objective | Key Constraints |
|---|---|---|
| Bull | Maximize return − risk_aversion × variance | Equity ≥ 45%, ≤ 80% |
| Bear | Minimize variance − return_reward × return | Equity ≤ 40%; gold+bond ≥ 60% |
| Crisis | Minimize variance | Equity ≤ 15%; gold ≥ 25%; bond ≥ 40% |

Ridge regularization is applied to the covariance matrix for PSD guarantee.
All allocations are long-only; no leverage.

In [ ]:
from regime_shift.portfolio import optimize_portfolio

asset_returns = prices[["equity", "gold", "bond"]].pct_change().iloc[1:]
est_returns = asset_returns.loc[:train_end].tail(config.portfolio_config.estimation_lookback)

port_sol = optimize_portfolio(
    regime=solution.regime,
    returns_through_date=est_returns,
    previous_weights=None,
    config=config.portfolio_config,
)
print(f"Regime: {port_sol.regime}")
print(f"Solver: {port_sol.solver} | Status: {port_sol.status}")
print(f"Weights: equity={port_sol.weights['equity']:.4f}, "
      f"gold={port_sol.weights['gold']:.4f}, "
      f"bond={port_sol.weights['bond']:.4f}")


### 9. Transaction Costs — Base Scenario (10 bps one-way)

Cost model: **base scenario** — 10 basis points one-way per asset.

- **Initial allocation:** turnover = Σ |w_i| (full L1).
- **Subsequent rebalance:** turnover = 0.5 × Σ |w_i − w_{prev}| (half-L1).
- **Net return:** r_net = (1 + r_gross) × (1 − cost) − 1.
- No cost on non-rebalance dates.

**Limitation:** These are assumed-cost scenarios, not measured historical spreads or fills. Market impact and capacity are excluded because volume/ADV data is absent. `^NSEI` is not a directly executable fill.

Alternative scenarios: optimistic (5 bps), stressed (20 bps).

### 10. Walk-Forward Backtest with NEXT_CLOSE

- **Evaluation observations:** 3732
- **Rebalance frequency:** 21 trading days
- **Minimum training window:** 126 observations
- **Regime counts (rebalance dates):** {'Bull': 1428, 'Bear': 1233, 'Crisis': 1071}
- **Total rebalances:** 178

In [ ]:
from regime_shift.backtest import run_walk_forward_backtest, run_benchmark
from regime_shift.benchmarks import static_60_40_weights, equal_weight_weights

strategy = run_walk_forward_backtest(
    prices=prices, config=config, risk_free_rate=0.0,
)
print(f"Backtest complete.")
print(f"Evaluation: {len(strategy.net_returns)} trading days")
print(f"  {strategy.net_returns.index[0].date()} to {strategy.net_returns.index[-1].date()}")
print(f"Rebalances: {strategy.rebalance_flags.sum()}")
for r in ["Bull", "Bear", "Crisis"]:
    print(f"  {r}: {strategy.regime_series.value_counts().get(r, 0)} days")


In [ ]:
bench_6040 = run_benchmark(
    prices=prices, weights=static_60_40_weights(), config=config,
    rebalance_flags=strategy.rebalance_flags,
    start_date=strategy.net_returns.index[0],
)
bench_ew = run_benchmark(
    prices=prices, weights=equal_weight_weights(), config=config,
    rebalance_flags=strategy.rebalance_flags,
    start_date=strategy.net_returns.index[0],
)
print("Benchmarks complete.")


### 11. Official Aligned Results

All strategies share the same evaluation dates, NEXT_CLOSE execution, base 10 bps cost model, and risk-free rate of 0.

In [ ]:
from regime_shift.metrics import compute_performance_metrics
import pandas as pd

def _row(label, net, gross, turn, costs):
    m = compute_performance_metrics(
        net, gross_returns=gross, turnover=turn, transaction_costs=costs, risk_free_rate=0.0)
    d = m.to_dict(); d["Strategy"] = label; return d

perf_rows = [
    _row("RegimeShift Gross", strategy.gross_returns, strategy.gross_returns,
         strategy.turnover, strategy.transaction_costs),
    _row("RegimeShift Net",   strategy.net_returns,   strategy.gross_returns,
         strategy.turnover, strategy.transaction_costs),
    _row("Static 60/40 Gross", bench_6040.gross_returns, bench_6040.gross_returns,
         bench_6040.turnover, bench_6040.transaction_costs),
    _row("Static 60/40 Net",   bench_6040.net_returns,   bench_6040.gross_returns,
         bench_6040.turnover, bench_6040.transaction_costs),
    _row("Equal Weight Gross", bench_ew.gross_returns, bench_ew.gross_returns,
         bench_ew.turnover, bench_ew.transaction_costs),
    _row("Equal Weight Net",   bench_ew.net_returns,   bench_ew.gross_returns,
         bench_ew.turnover, bench_ew.transaction_costs),
]
perf_df = pd.DataFrame(perf_rows)
display(perf_df[["Strategy","CAGR","Sharpe","Sortino","Maximum Drawdown",
                  "Calmar","Annualised Turnover","Transaction Cost Drag"]])


### 12. Ablation Study Results

| strategy | sharpe | cagr | max_drawdown | ann_turnover |
|---|---|---|---|---|
| A_RegimeShift_Full | 0.3999 | 0.0602 | 0.3577 | 3.3180 |
| B_NoRegime_Optimizer | 0.4567 | 0.0755 | 0.3443 | 1.0280 |
| C_VolRule | 0.6262 | 0.0761 | 0.2549 | 0.5280 |
| D_HMM_Fixed | 0.6347 | 0.0666 | 0.1961 | 1.8341 |
| E_MinVariance | 0.9944 | 0.0514 | 0.1137 | 0.1036 |
| F_Static_6040 | 0.7595 | 0.0720 | 0.2339 | 0.1677 |
| G_EqualWeight | 0.5883 | 0.0895 | 0.3296 | 0.1981 |

Ablation strategies: (A) Full RegimeShift HMM+CVXPY, (B) No-regime optimizer (always Bull), (C) Volatility-rule allocation, (D) HMM+fixed weights, (E) Minimum variance, (F) Static 60/40, (G) Equal Weight. All share identical dates, costs, and execution.

### 13. Chronological Subperiod Analysis

> **Important:** The retrospective evaluation period (2022-2026) is **not** a pristine holdout because the full historical sample was inspected during earlier project development.

| period | n_obs | cagr | sharpe | max_drawdown |
|---|---|---|---|---|
| Development | 1860 | 0.0227 | 0.2994 | 0.1910 |
| Validation | 742 | 0.0900 | 0.4017 | 0.3506 |
| Retrospective Evaluation | 1130 | 0.1046 | 1.0792 | 0.1069 |


### 14. Rolling-Origin Evaluation

| fold | n_obs | cagr | sharpe | max_drawdown |
|---|---|---|---|---|
| 2013-2015 | 715 | -0.0074 | -0.0461 | 0.1399 |
| 2016-2018 | 740 | 0.0226 | 0.3649 | 0.1063 |
| 2019-2021 | 742 | 0.0900 | 0.4017 | 0.3506 |
| 2022-2024 | 739 | 0.0509 | 0.6581 | 0.1043 |
| 2025-2026 | 391 | 0.2135 | 1.6596 | 0.1069 |


### 15. Bootstrap Uncertainty (Paired Moving-Block, n=2000, block=21)

| metric | mean_diff | ci_lower_95 | ci_upper_95 | zero_inside_ci | statistically_significant | n_valid | note |
|---|---|---|---|---|---|---|---|
| RegimeShift_sharpe_vs_6040 | -0.2354 | -0.7971 | 0.3960 | True | False | 2000 | No significance claimed when zero lies inside CI |
| RegimeShift_sharpe_vs_EW | -0.2673 | -0.7890 | -0.0081 | False | True | 2000 | Zero outside CI; causal retrospective data only |
| RegimeShift_cagr_vs_6040 | -0.0077 | -0.0666 | 0.0556 | True | False | 2000 | No significance claimed when zero lies inside CI |
| RegimeShift_cagr_vs_EW | -0.0284 | -0.0619 | 0.0065 | True | False | 2000 | No significance claimed when zero lies inside CI |
| RegimeShift_max_drawdown_vs_6040 | 0.0770 | -0.1648 | 0.2920 | True | False | 2000 | No significance claimed when zero lies inside CI |
| RegimeShift_max_drawdown_vs_EW | 0.0391 | -0.0343 | 0.1486 | True | False | 2000 | No significance claimed when zero lies inside CI |

No significance is claimed when zero lies inside a confidence interval. All results are retrospective causal simulation data only.

### 16. HMM Stability Diagnostics

- Total rolling HMM fits: **178**
- Convergence rate: **100.0%**
- Median log-likelihood: **-1809.41**
- State occupancy (Bull): **38.2%**
- State occupancy (Bear): **33.1%**
- State occupancy (Crisis): **28.7%**


### 17. Negative Findings

**RegimeShift did not outperform the simpler static benchmarks on headline risk-adjusted performance.**

Under corrected NEXT_CLOSE causal timing and base 10 bps transaction costs:

| Strategy | CAGR | Sharpe | Max Drawdown |
|---|---|---|---|
| RegimeShift | ~6.0% | ~0.40 | ~35.8% |
| Static 60/40 | ~6.6% | ~0.70 | ~23.4% |
| Equal Weight | ~8.8% | ~0.57 | ~33.0% |

(Read exact numbers from `results/submission/performance_summary.csv`.)

**What this research demonstrated:**
- The causal pipeline design (rolling train-only scaler+HMM, NEXT_CLOSE ordering) is leak-free and reproducible.
- HMM regime detection produces interpretable Bull/Bear/Crisis states with reasonable transition persistence.
- The added complexity of dynamic regime allocation does not generate alpha over static equal-weight or 60/40 portfolios in this Indian multi-asset dataset.
- High turnover (3.3× annualised) relative to static benchmarks (~0.18×) creates significant cost drag that offsets any gross diversification benefit.
- Bootstrap confidence intervals for paired Sharpe differences generally include zero, consistent with no statistically significant outperformance.

### 18. Limitations

1. **Index-level equity simulation:** `^NSEI` is an index, not a directly executable instrument. A real portfolio requires an equity ETF with verified common history, corporate-action treatment, and liquidity audit.
2. **Assumed costs, not measured:** 5/10/20 bps are scenario assumptions. Market impact, bid-ask spreads, and capacity are not modelled.
3. **Short-duration bond proxy:** LIQUIDBEES.NS provides no long-term duration hedge or inflation protection.
4. **Retrospective evaluation:** All periods were visible during development. No segment is a pristine holdout.
5. **Three assets only:** Broader diversification requires expanding the universe.
6. **Gaussian HMM:** Student-t emissions may better capture fat tails.
7. **No alpha proven:** Positive CAGR reflects the underlying market uptrend, not superior risk-adjusted returns.

### 19. Reproduction

```bash
pip install -e ".[dev]"

# Official experiment
python run_submission.py \
    --data-path data/submission_market_data.csv \
    --cost-scenario base \
    --output-dir results/submission

# Research validation suite
python scripts/run_research_suite.py

# Rebuild and execute this notebook
python scripts/build_notebook.py
python -m jupyter nbconvert --to notebook --execute \
    notebooks/RegimeShift_Submission.ipynb --inplace \
    --ExecutePreprocessor.timeout=3600

# Tests
python -m pytest -q
python scripts/verify_release.py
```